In [ ]:
import math
import time
from dataclasses import dataclass

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

In [ ]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

In [ ]:
@dataclass(frozen=True)
class ShapeSpec:
    b: int
    c: int
    h: int
    w: int


SIZES  = (2, 4, 8, 16, 32, 64)
SHAPES = [ShapeSpec(b=b, c=c, h=hw, w=hw) for b in SIZES for c in SIZES for hw in SIZES]
DEVICE = get_device()
DTYPE  = torch.float16 if (DEVICE.type == "cuda") else torch.float32

print(f"[device]: {DEVICE}")

In [ ]:
def _pearsonr_flat(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    x = x.flatten(1)
    y = y.flatten(1)
    x = x - x.mean(dim=-1, keepdim=True)
    y = y - y.mean(dim=-1, keepdim=True)
    num = (x * y).sum(dim=-1)
    den = x.norm(dim=-1) * y.norm(dim=-1)
    return num / den.clamp_min(1e-8)

In [ ]:
def _offsets(kernel_radius: int, device: torch.device) -> torch.Tensor:
    r = int(kernel_radius)
    dh = torch.arange(-r, r + 1, device=device)
    dw = torch.arange(-r, r + 1, device=device)
    grid_dh, grid_dw = torch.meshgrid(dh, dw, indexing="ij")
    offsets = torch.stack([grid_dh.flatten(), grid_dw.flatten()], dim=-1)
    offsets = offsets[(offsets[:, 0] != 0) | (offsets[:, 1] != 0)]
    return offsets


def max_fn(fm: torch.Tensor, kernel_radius: int) -> torch.Tensor:
    b, c, h, w = fm.shape
    offsets = _offsets(kernel_radius, fm.device)
    best = fm.new_full((), -float("inf"))
    for c1 in range(c):
        x0 = fm[:, c1]
        for c2 in range(c1 + 1, c):
            y0 = fm[:, c2]
            for dh, dw in offsets.tolist():
                dh = int(dh)
                dw = int(dw)
                hs = max(0, -dh)
                he = min(h, h - dh)
                ws = max(0, -dw)
                we = min(w, w - dw)
                if (he <= hs) or (we <= ws):
                    continue
                x = x0[:, hs:he, ws:we]
                y = y0[:, hs + dh : he + dh, ws + dw : we + dw]
                sim = _pearsonr_flat(x, y).max()
                best = torch.maximum(best, sim)
    return best

In [ ]:
def luca_fn(fm: torch.Tensor, kernel_radius: int) -> torch.Tensor:
    b, c, h, w = fm.shape
    k = 2 * int(kernel_radius) + 1
    k2 = k * k
    patches = F.unfold(fm.reshape(b * c, 1, h, w), kernel_size=k, padding=kernel_radius)
    patches = patches.reshape(b, c, k2, h * w)
    center_idx = k2 // 2
    center = patches[:, :, center_idx, :]
    start = (k2 // 2) + 1
    neigh = patches[:, :, start:, :]
    bl = b * h * w
    v = center.permute(1, 0, 2).reshape(c, bl)
    v = v - v.mean(dim=-1, keepdim=True)
    v = v / v.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    n = neigh.permute(1, 2, 0, 3).reshape(c, k2 - start, bl)
    n = n - n.mean(dim=-1, keepdim=True)
    n = n / n.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    sims = torch.einsum("ib,job->ijo", v, n)
    sims = sims.max(dim=-1).values
    mask = torch.triu(torch.ones((c, c), device=fm.device, dtype=torch.bool), diagonal=1)
    sims = sims.masked_fill(~mask, -float("inf"))
    return sims.max()

In [ ]:
def vmap_fn(fm: torch.Tensor, kernel_radius: int) -> torch.Tensor:
    b, c, h, w = fm.shape
    offsets = _offsets(kernel_radius, fm.device)
    vmap_impl = getattr(torch, "vmap", None)
    if vmap_impl is None:
        vmap_impl = torch.func.vmap

    def per_c1(c1) -> torch.Tensor:
        c1_int = int(c1.item()) if isinstance(c1, torch.Tensor) else int(c1)
        x0 = fm[:, c1_int]
        best_local = fm.new_full((), -float("inf"))
        for c2 in range(c1_int + 1, c):
            y0 = fm[:, c2]
            for dh, dw in offsets.tolist():
                dh = int(dh)
                dw = int(dw)
                hs = max(0, -dh)
                he = min(h, h - dh)
                ws = max(0, -dw)
                we = min(w, w - dw)
                if (he <= hs) or (we <= ws):
                    continue
                x = x0[:, hs:he, ws:we]
                y = y0[:, hs + dh : he + dh, ws + dw : we + dw]
                sim = _pearsonr_flat(x, y).max()
                best_local = torch.maximum(best_local, sim)
        return best_local

    c1s = torch.arange(c, device=fm.device)
    bests = vmap_impl(per_c1)(c1s)
    bests = bests.masked_fill(c1s == (c - 1), -float("inf"))
    return bests.max()

In [ ]:
def _sync_if_cuda(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def time_fn(fn, fm: torch.Tensor, kernel_radius: int, warmup: int = 2, runs: int = 10) -> float:
    for _ in range(warmup):
        _ = fn(fm, kernel_radius)
    _sync_if_cuda(fm.device)
    times: list[float] = []
    for _ in range(runs):
        t0 = time.perf_counter()
        _ = fn(fm, kernel_radius)
        _sync_if_cuda(fm.device)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    return float(sum(times) / max(1, len(times)))


def make_fm(shape: ShapeSpec, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    fm = torch.randn((shape.b, shape.c, shape.h, shape.w), device=device, dtype=dtype)
    return fm


def benchmark(
    shapes: list[ShapeSpec],
    kernel_radius: int = 2,
    device: torch.device | None = None,
    dtype: torch.dtype | None = None,
    warmup: int = 2,
    runs: int = 10,
) -> list[dict]:
    device = device or DEVICE
    dtype = dtype or DTYPE
    fns = {"max_fn": max_fn, "luca_fn": luca_fn, "vmap_fn": vmap_fn}
    results: list[dict] = []
    for shape in tqdm(shapes):
        fm = make_fm(shape, device=device, dtype=dtype)
        row = {"b": shape.b, "c": shape.c, "h": shape.h, "w": shape.w}
        for name, fn in fns.items():
            row[name] = time_fn(fn, fm, kernel_radius=kernel_radius, warmup=warmup, runs=runs)
        results.append(row)
    return results

In [ ]:
KERNEL_RADIUS = 2
WARMUP = 1
RUNS = 10

results = benchmark(SHAPES, kernel_radius=KERNEL_RADIUS, warmup=WARMUP, runs=RUNS)

fastest = sorted(results, key=lambda r: min(r["max_fn"], r["luca_fn"], r["vmap_fn"]))[:10]
slowest = sorted(results, key=lambda r: max(r["max_fn"], r["luca_fn"], r["vmap_fn"]), reverse=True)[:10]

fastest, slowest[:3]